## ResNet18 - ImageNet final model

In [4]:
import torch
import torch.nn as nn
import numpy as np
import random
import time
import matplotlib.pyplot as plt

from pathlib import Path
from torchvision import datasets
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# Reproducibility
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# Device
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

# Dataset paths
data_path = Path("../../data")

# Same ImageNet preprocessing as before
weights = ResNet18_Weights.DEFAULT
transform = weights.transforms()

train_dataset = datasets.ImageFolder(
    root=data_path / "train",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root=data_path / "test",
    transform=transform
)

# DataLoaders
train_loader_ft = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

test_loader_ft = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

# Fresh pretrained model
model_ft = resnet18(weights=weights)

# Freeze all pretrained layers
for param in model_ft.parameters():
    param.requires_grad = False

# Replace the ImageNet classifier with 10 EuroSAT outputs
model_ft.fc = nn.Linear(
    model_ft.fc.in_features,
    10
)

# Unfreeze layer4
for param in model_ft.layer4.parameters():
    param.requires_grad = True

# The new fc layer is already trainable
model_ft = model_ft.to(device)

# Loss function
criterion_ft = nn.CrossEntropyLoss()

# Optimiser
optimizer_ft = torch.optim.Adam([
    {"params": model_ft.layer4.parameters(), "lr": 0.001},
    {"params": model_ft.fc.parameters(), "lr": 0.001}
])

print("Device:", device)
print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Batches per epoch:", len(train_loader_ft))

print("Trainable parameters:", sum(
    p.numel()
    for p in model_ft.parameters()
    if p.requires_grad
))

Device: mps
Training images: 21600
Test images: 5400
Batches per epoch: 675
Trainable parameters: 8398858


In [2]:
def train_finetuning(
    model,
    train_loader,
    criterion,
    optimizer,
    device,
    epochs=5
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "epoch_time": []
    }

    total_start = time.perf_counter()

    for epoch in range(epochs):

        model.train()

        # Keep pretrained BatchNorm statistics fixed
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()

        running_loss = 0.0
        correct = 0
        total = 0

        epoch_start = time.perf_counter()

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            # Forward pass
            outputs = model(images)

            # Loss
            loss = criterion(outputs, labels)

            # Backpropagation
            loss.backward()
            optimizer.step()

            # Metrics
            running_loss += loss.item() * labels.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        # Synchronise GPU before timing
        if device.type == "mps":
            torch.mps.synchronize()
        elif device.type == "cuda":
            torch.cuda.synchronize()

        epoch_time = time.perf_counter() - epoch_start

        train_loss = running_loss / total
        train_accuracy = 100 * correct / total

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["epoch_time"].append(epoch_time)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Loss: {train_loss:.4f} | "
            f"Training accuracy: {train_accuracy:.2f}% | "
            f"Time: {epoch_time / 60:.2f} minutes",
            flush=True
        )

    total_time = time.perf_counter() - total_start

    print("\nTraining completed!")
    print(f"Total time: {total_time / 60:.2f} minutes")
    print(
        f"Average time per epoch: "
        f"{np.mean(history['epoch_time']) / 60:.2f} minutes"
    )

    return history, total_time

In [ ]:
history_ft_5epochs, training_time_ft_5epochs = train_finetuning(
    model_ft,
    train_loader_ft,
    criterion_ft,
    optimizer_ft,
    device,
    epochs=5
)

In [ ]:
torch.save({
    "epoch": 5,
    "model_state_dict": model_ft.state_dict(),
    "optimizer_state_dict": optimizer_ft.state_dict(),
    "history": history_ft_5epochs,
    "training_time": training_time_ft_5epochs,
    "seed": 42,
    "batch_size": 32,
    "layer4_lr": 0.0001,
    "fc_lr": 0.001
}, "resnet18_layer4_fc_5epochs_fresh.pth")

print("Checkpoint saved!")

In [ ]:
def test_finetuned_model(
    model,
    test_loader,
    criterion,
    device
):
    model.eval()

    all_predictions = []
    all_labels = []

    total_loss = 0.0
    correct = 0
    total = 0

    start_time = time.perf_counter()

    with torch.inference_mode():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            predictions = outputs.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    # Synchronise for accurate timing
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

    test_time = time.perf_counter() - start_time

    test_loss = total_loss / total
    test_accuracy = 100 * correct / total

    print("\nFINAL TEST RESULTS")
    print("=" * 45)
    print(f"Test images: {total}")
    print(f"Test accuracy: {test_accuracy:.2f}%")
    print(f"Test loss: {test_loss:.4f}")
    print(f"Testing time: {test_time:.2f} seconds")

    # Class names
    class_names = test_loader.dataset.classes

    # Normalised confusion matrix
    cm = confusion_matrix(
        all_labels,
        all_predictions,
        labels=range(len(class_names)),
        normalize="true"
    ) * 100

    fig, ax = plt.subplots(figsize=(12, 10))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    disp.plot(
        ax=ax,
        cmap="Blues",
        xticks_rotation=45,
        values_format=".1f"
    )

    ax.set_title(
        "ResNet-18: Layer4 + FC Fine-Tuning (5 Epochs)",
        fontsize=15
    )

    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("Actual Class")

    plt.tight_layout()

    plt.savefig(
        "resnet18_layer4_fc_5epochs_fresh_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("\nCLASSIFICATION REPORT")
    print("=" * 45)

    print(
        classification_report(
            all_labels,
            all_predictions,
            labels=range(len(class_names)),
            target_names=class_names,
            zero_division=0
        )
    )

    return {
        "test_accuracy": test_accuracy,
        "test_loss": test_loss,
        "test_time": test_time,
        "predictions": all_predictions,
        "labels": all_labels
    }

In [ ]:
results_ft_5epochs_fresh = test_finetuned_model(
    model_ft,
    test_loader_ft,
    criterion_ft,
    device
)